# Calibrated Explanations for multiclass Classification
## Stability and Robustness

### 1. Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import pickle
import numpy as np
from scipy import stats as st
import pandas as pd
import warnings
import Multi_Class_Experiments_rob as mcer
import Multi_Class_Experiments_stab as mces
warnings.filterwarnings("ignore")

In [ ]:
columns = ["dataset","Model","CE", "CCE", "L C", "S C", "L U", "S U"]
df_all_stabilities =  pd.DataFrame(columns=columns)
df_all_stability_time =  pd.DataFrame(columns=columns)
df_all_rob_time =  pd.DataFrame(columns=columns)
df_all_rob =  pd.DataFrame(columns=columns)


In [ ]:

data_characteristics = {'CIA': 9, 
                        'Azure': 30, 
                        'AI4I': 6, 
                        'Pump': 53}
a_names = {'xGB':'XGB', 'RF':'RF', 'DT':'DT', "AAGru":'GRU'}
#'stab_timer'



### 2 Tests execution
execute robustness and stability  tests to save the results as pickled files
run once

In [ ]:
mcer.test_robustness()
mces.test_stability()

### Stability, Robustness and time dataset-analysis function


In [ ]:
def analyze_dataset_stab(results,dset):    

    stab_rank = {}
    stab_val = {}
    average_results = {}
    for a in ['xGB', 'RF', 'DT', "AAGru"]:
        average_results[a+'_stab_ce'] = {}
        average_results[a+'_stab_cce'] = {}
        average_results[a+'_stab_lime'] = {}
        average_results[a+'_stab_lime_va'] = {}
        average_results[a+'_stab_shap'] = {}
        average_results[a+'_stab_shap_va'] = {}


    n = results['test_size']
    r = results['num_rep']
    df_stability =  pd.DataFrame(columns=columns)
    for d in np.sort([k for k in results.keys()]):
        if d in ['test_size', 'num_rep']:
            continue
        algorithms = results[d].keys()
        for a in algorithms:
            stability = results[d][a]["stability"]
            num_classes = len(stability['ce'][0][0])
            for key in ['ce', 'cce']:    
                ranks = []
                for j in range(n):
                    rank = []
                    for i in range(r):
                        for c in range(num_classes):
                            rank.append(np.argsort(np.abs(stability[key][i][j][c]['predict']))[-1:][0])
                        ranks.append(rank)
                stab_rank[key] = st.mode(ranks, axis=1)[0] # Find most important feature per instance
                value = []
                for j in range(n):
                    class_value=[]
                    for c in range(num_classes):
                        class_value.append([np.mean([stability[key][i][j][c]['predict'][stab_rank[key][j]] for i in range(r)]), np.var([stability[key][i][j][c]['predict'][stab_rank[key][j]] for i in range(r)])])
                    value.append([np.mean([m[0] for m in class_value]),np.mean([v[1] for v in class_value])])
                stab_val[key] = value 

            stability = results[d][a]['stability']
                        
            for key in ['lime', 'lime_va', 'shap', 'shap_va']:   
                ranks = []
                for j in range(n):
                    rank = []
                    for i in range(r):
                        rank.append(np.argsort(np.abs(stability[key][i][j]))[-1:][0])
                    ranks.append(rank)
                stab_rank[key] = st.mode(ranks, axis=1)[0] # Find most important feature per instance
                value = []
                for j in range(n):
                    value.append([np.mean([stability[key][i][j][stab_rank[key][j]] for i in range(r)]), np.var([stability[key][i][j][stab_rank[key][j]] for i in range(r)])])
                stab_val[key] = value 
            for m ,_ in enumerate(stab_val['ce']):        
                df_stability = pd.concat([df_stability,pd.DataFrame([{"dataset":dset,
                                    "Model":a,
                                    "CE":stab_val['ce'][m][1],
                                    "CCE":stab_val['cce'][m][1],
                                    "L C":stab_val['lime_va'][m][1],
                                    "S C":stab_val['shap_va'][m][1],
                                    "L U":stab_val['lime'][m][1],
                                    "S U":stab_val['shap'][m][1] }])], ignore_index= True)
            
            
            for key in ['ce', 'cce', 'lime', 'lime_va', 'shap', 'shap_va']:
                average_results[a+'_stab_'+key][d] = [t[1] for t in stab_val[key]]
            
    for a in algorithms:
        #print("\hline")
        print(f"&{a_names[a]}", end="")
        for key in ['ce', 'cce', 'lime_va', 'shap_va', 'lime', 'shap']:
            print(f' & {np.mean([ np.mean(v) for v in average_results[a+"_stab_"+key].values()]):.0e}',end='')
        print("  \\\\\cline{2-8}")

    print("\hline")


In [ ]:
def analyze_dataset_runtime(results,dset):  

    n = results['test_size']
    average_time = {}
    average_time['num_features'] = []
    for a in ['xGB', 'RF', 'DT', "AAGru"]:
        average_time[a+'_stab_ce'] = []
        average_time[a+'_stab_cce'] = []
        average_time[a+'_stab_lime'] = []
        average_time[a+'_stab_lime_va'] = []
        average_time[a+'_stab_shap'] = []
        average_time[a+'_stab_shap_va'] = []
    columns = ["dataset","Model","CE", "CCE", "L C", "S C", "L U", "S U"]
    df_stability_time =  pd.DataFrame(columns=columns)
    for d in np.sort([k for k in results.keys()]):
        if d in ['test_size', 'num_rep']:
            continue
        for a in results[d]:
            r_time = results[d][a]['stab_timer']
            average_time[a+'_stab_ce'].append(np.mean([t/n for t in r_time["ce"]]))
            average_time[a+'_stab_cce'].append(np.mean([t/n for t in r_time["cce"]]))
            average_time[a+'_stab_lime'].append(np.mean([t/n for t in r_time["lime"]]))
            average_time[a+'_stab_lime_va'].append(np.mean([t/n for t in r_time["lime_va"]]))
            average_time[a+'_stab_shap'].append(np.mean([t/n for t in r_time["shap"]]))
            average_time[a+'_stab_shap_va'].append(np.mean([t/n for t in r_time["shap_va"]]))
            average_time['num_features'].append(data_characteristics[d])
    
    for a in results[dset]:
        r_time = results[dset][a]['stab_timer']
        for m,_ in enumerate(r_time["lime_va"]):
            df_stability_time = pd.concat([df_stability_time,pd.DataFrame([{"dataset":dset,
                        "Model":a,
                        "CE":r_time['ce'][m],
                        "CCE":r_time['cce'][m],
                        "L C":r_time['lime_va'][m],
                        "S C":r_time['shap_va'][m],
                        "L U":r_time['lime'][m],
                        "S U":r_time['shap'][m] }])], ignore_index=True)
    for a in ['xGB', 'RF', 'DT', "AAGru"]:
        print(f"&{a_names[a]}", end="")
        for key in ['ce', 'cce', 'lime_va', 'shap_va', 'lime', 'shap']:
            print(f' & {np.mean(average_time[a+"_stab_"+key]):.4f}  ',end='')
        print("  \\\\\cline{2-8}")

    print("\hline")
    df_stability_time.to_csv("evaluation/stability_time_"+dset+".csv", index=False) 
    return df_stability_time 



In [ ]:
def analyze_dataset_rob(results,dset):
    rob_rank = {}
    rob_val = {}
    rob_proba = []
    rob_proba_va = []
    average_results = {}
    df_rob =  pd.DataFrame(columns=columns)

    for a in ['xGB', 'RF', 'DT', "AAGru"]:
        average_results[a+'_rob_ce'] = {}
        average_results[a+'_rob_cce'] = {}
        average_results[a+'_rob_lime'] = {}
        average_results[a+'_rob_lime_va'] = {}
        average_results[a+'_rob_shap'] = {}
        average_results[a+'_rob_shap_va'] = {}

    n = results['test_size']
    r = results['num_rep']


    for d in np.sort([k for k in results.keys()]):
        if d in ['test_size', 'num_rep']:
            continue
        algorithms = results[d].keys()
        for a in algorithms:
            robustness = results[d][a]['robustness']
            for key in ['ce', 'cce']:               
                num_classes = len(robustness[key][0][0])
                ranks = []
                values = []
                for j in range(n):
                    rank = []
                    value = []
                    for i in range(r):
                        for c in range(len(robustness[key][i][j])):
                            rank.append(np.argsort(np.abs(robustness[key][i][j][c]['predict']))[-1:][0])
                        ranks.append(rank)
                    values.append(value)
                rob_rank[key] = st.mode(ranks, axis=1)[0] # Find most important feature per instance
                value = []
                for j in range(n):
                    for c in range(num_classes):
                        var = []
                        mean = []
                        for i in range(r):
                            if c < len(robustness[key][i][j]):
                                mean.append(robustness[key][i][j][c]['predict'][rob_rank[key][j]])
                                var.append(robustness[key][i][j][c]['predict'][rob_rank[key][j]])
                        value.append([np.mean(mean), np.var(var)])
                rob_val[key] = value

            robustness = results[d][a]['robustness']
                
            for key in ['lime', 'lime_va', 'shap', 'shap_va']:    
                ranks = []
                for j in range(n):
                    rank = []
                    for i in range(r):
                        rank.append(np.argsort(np.abs(robustness[key][i][j]))[-1:][0])
                    ranks.append(rank)
                rob_rank[key] = st.mode(ranks, axis=1)[0] # Find most important feature per instance
                value = []
                for j in range(n):
                    value.append([np.mean([robustness[key][i][j][rob_rank[key][j]] for i in range(r)]), np.var([robustness[key][i][j][rob_rank[key][j]] for i in range(r)])])
                rob_val[key] = value 

            for m ,_ in enumerate(rob_val['lime_va']):        
                df_rob = pd.concat([df_rob,pd.DataFrame([{"dataset":d,
                                    "Model": a,
                                    "CE":rob_val['ce'][m][1],
                                    "CCE":rob_val['cce'][m][1],
                                    "L C":rob_val['lime_va'][m][1],
                                    "S C":rob_val['shap_va'][m][1],
                                    "L U":rob_val['lime'][m][1],
                                    "S U":rob_val['shap'][m][1] }])], ignore_index=True)

            for inst in range(n):
                rob_proba.append(np.var([robustness['proba'][j][inst] for j in range(r)]))
                rob_proba_va.append(np.var([robustness['proba_va'][j][inst] for j in range(r)]))

            for key in ['ce', 'cce', 'lime', 'lime_va', 'shap', 'shap_va']:
                average_results[a+'_rob_'+key][d] = np.mean([t[1] for t in rob_val[key]])
            
    for a in algorithms:
        print(f"&{a_names[a]}", end="")
        for key in ['ce', 'cce', 'lime_va', 'shap_va', 'lime', 'shap']:
            print(f' & {np.mean([v for v in average_results[a+"_rob_"+key].values()]):.4f}',end='')
        print("  \\\\\cline{2-8}")
    print("\hline")
    df_rob.to_csv("evaluation/rob_"+dset+".csv", index=False)  
    return df_rob


In [ ]:
def analyze_dataset_runtime_rob(results,dset):
    n = results['test_size']
    average_time = {}
    average_time['num_features'] = []
    df_rob_time =  pd.DataFrame(columns=columns)

    for a in ['xGB', 'RF', 'DT', "AAGru"]:
        average_time[a+'_rob_ce'] = []
        average_time[a+'_rob_cce'] = []
        average_time[a+'_rob_lime'] = []
        average_time[a+'_rob_lime_va'] = []
        average_time[a+'_rob_shap'] = []
        average_time[a+'_rob_shap_va'] = []

    for d in np.sort([k for k in results.keys()]):
        if d in ['test_size', 'num_rep']:
            continue

        for a in results[d]:
            r_time = results[d][a]['rob_timer']
            average_time[a+'_rob_ce'].append(np.mean([t/n for t in r_time["ce"]]))
            average_time[a+'_rob_cce'].append(np.mean([t/n for t in r_time["cce"]]))
            average_time[a+'_rob_lime'].append(np.mean([t/n for t in r_time["lime"]]))
            average_time[a+'_rob_lime_va'].append(np.mean([t/n for t in r_time["lime_va"]]))
            average_time[a+'_rob_shap'].append(np.mean([t/n for t in r_time["shap"]]))
            average_time[a+'_rob_shap_va'].append(np.mean([t/n for t in r_time["shap_va"]]))
            average_time['num_features'].append(data_characteristics[d])
        
    for a in results[dset]:
        r_time = results[dset][a]['rob_timer']
        for m,_ in enumerate(r_time["ce"]):
            df_rob_time = pd.concat([df_rob_time,pd.DataFrame([{"dataset":dset,
                        "Model":a,
                        "CE":r_time['ce'][m],
                        "CCE":r_time['cce'][m],
                        "L C":r_time['lime_va'][m],
                        "S C":r_time['shap_va'][m],
                        "L U":r_time['lime'][m],
                        "S U":r_time['shap'][m] }])], ignore_index=True)
    for a in ['xGB', 'RF', 'DT', "AAGru"]:
        print(f"&{a_names[a]}", end="")
        for key in ['ce', 'cce', 'lime_va', 'shap_va', 'lime', 'shap']:
            print(f' & {np.mean(average_time[a+"_rob_"+key]):.4f}  ',end='')
        print("  \\\\\cline{2-8}")
    print("\hline")

    df_rob_time.to_csv("evaluation/rob_time_"+dset+".csv", index=False)  
    return df_rob_time


In [ ]:
for i, dataset in enumerate(data_characteristics):
    with open('evaluation/results_stab_all_' + dataset + '.pkl', 'rb') as f:
        results = pickle.load(f)
    if i == 0:
        print('Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\\\')
        print('\hline')
    print('\multirow{4}{*}{'+dataset+'}')
    df_all_stabilities =  pd.concat([df_all_stabilities,analyze_dataset_stab(results,dataset)])

    

Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\
\hline
\multirow{4}{*}{CIA}
&XGB & 4e-33 & 4e-34 & 5e-05 & 2e-32 & 7e-07 & 3e-37  \\\cline{2-8}
&RF & 7e-33 & 9e-34 & 2e-05 & 4e-35 & 1e-06 & 1e-35  \\\cline{2-8}
&DT & 1e-35 & 1e-35 & 4e-08 & 2e-35 & 3e-05 & 3e-34  \\\cline{2-8}
&GRU & 3e-33 & 3e-33 & 2e-03 & 1e-31 & 2e-07 & 2e-35  \\\cline{2-8}
\hline
\multirow{4}{*}{Azure}
&XGB & 1e-33 & 2e-34 & 1e-03 & 8e-31 & 1e-05 & 6e-15  \\\cline{2-8}
&RF & 3e-33 & 3e-34 & 6e-04 & 3e-31 & 1e-05 & 9e-10  \\\cline{2-8}
&DT & 9e-34 & 4e-34 & 6e-04 & 3e-31 & 5e-05 & 4e-09  \\\cline{2-8}
&GRU & 6e-33 & 1e-33 & 2e-03 & 8e-31 & 2e-05 & 8e-09  \\\cline{2-8}
\hline
\multirow{4}{*}{AI4I}
&XGB & 2e-33 & 2e-34 & 6e-04 & 3e-31 & 3e-06 & 2e-35  \\\cline{2-8}
&RF & 4e-33 & 5e-34 & 5e-04 & 2e-31 & 3e-06 & 1e-35  \\\cline{2-8}
&DT & 8e-34 & 3e-35 & 5e-04 & 2e-31 & 2e-05 & 2e-34  \\\cline{2-8}
&GRU & 9e-33 & 2e-33 & 1e-03 & 3e-31 & 2e-06 & 2e-35  \\\cline{2-8}
\hline
\multirow{4}{*}{Pump}
&XGB & 2e-33 & 1e-34

As can be seen above, the stability is practically 0 for both factual CE (CE) and Alternative CE (ACE), illustrating that the method is stable by definition. Explanations extracted using SHAP (S_C)  from calibrated models and SHAP (S_U) from  uncalibrated models are also practically 0. LIME on calibrated models (L_C) and  LIME (L_U)  on uncalibrated models are clearly less stable.

In [ ]:
for i, dataset in enumerate(data_characteristics):
    with open('evaluation/results_rob_all_' + dataset + '.pkl', 'rb') as f:
        results = pickle.load(f)
    if i == 0:
        print('Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\\\')
        print('\hline')
    print('\multirow{4}{*}{'+dataset+'}')
    df_all_rob =  pd.concat([df_all_rob,analyze_dataset_rob(results,dataset)])


Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\
\hline
\multirow{4}{*}{CIA}
&XGB & 0.0079 & 0.0068 & 0.0233 & 0.0119 & 0.0002 & 0.0001  \\\cline{2-8}
&RF & 0.0052 & 0.0037 & 0.0144 & 0.0087 & 0.0001 & 0.0000  \\\cline{2-8}
&DT & 0.0087 & 0.0077 & 0.0109 & 0.0066 & 0.0039 & 0.0037  \\\cline{2-8}
&GRU & 0.0054 & 0.0054 & 0.0142 & 0.0088 & 0.0002 & 0.0001  \\\cline{2-8}
\hline
\multirow{4}{*}{Azure}
&XGB & 0.0028 & 0.0025 & 0.1493 & 0.0767 & 0.0001 & 0.0001  \\\cline{2-8}
&RF & 0.0021 & 0.0016 & 0.1105 & 0.0263 & 0.0001 & 0.0000  \\\cline{2-8}
&DT & 0.0030 & 0.0027 & 0.1093 & 0.0256 & 0.0025 & 0.0013  \\\cline{2-8}
&GRU & 0.0028 & 0.0027 & 0.1170 & 0.0545 & 0.0001 & 0.0000  \\\cline{2-8}
\hline
\multirow{4}{*}{AI4I}
&XGB & 0.0042 & 0.0038 & 0.2220 & 0.0603 & 0.0001 & 0.0001  \\\cline{2-8}
&RF & 0.0030 & 0.0022 & 0.1653 & 0.0259 & 0.0001 & 0.0000  \\\cline{2-8}
&DT & 0.0045 & 0.0041 & 0.1636 & 0.0249 & 0.0020 & 0.0019  \\\cline{2-8}
&GRU & 0.0030 & 0.0029 & 0.1737 & 0.0271 & 0.0001 &

In [ ]:
for i, dataset in enumerate(data_characteristics):
    with open('evaluation/results_stab_all_' + dataset + '.pkl', 'rb') as f:
        results = pickle.load(f)
    if i == 0:
        print('Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\\\')
        print('\hline')
    print('\multirow{4}{*}{'+dataset+'}')
    df_all_stability_time =  pd.concat([df_all_stability_time,analyze_dataset_runtime(results,dataset)])


Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\
\hline
\multirow{4}{*}{CIA}
&XGB & 0.0467   & 0.0645   & 0.0308   & 0.0108   & 0.0802   & 0.0575    \\\cline{2-8}
&RF & 0.0281   & 0.0441   & 0.0295   & 0.0066   & 0.0638   & 0.2037    \\\cline{2-8}
&DT & 0.0224   & 0.0368   & 0.0301   & 0.0064   & 0.0388   & 0.0080    \\\cline{2-8}
&GRU & 0.0302   & 0.0407   & 0.0322   & 0.0082   & 0.0522   & 0.0663    \\\cline{2-8}
\hline
\multirow{4}{*}{Azure}
&XGB & 0.5534   & 0.8094   & 0.0510   & 0.0155   & 0.1256   & 0.0687    \\\cline{2-8}
&RF & 0.5363   & 0.8090   & 0.0495   & 0.0111   & 0.1037   & 0.2028    \\\cline{2-8}
&DT & 0.5166   & 0.7883   & 0.0531   & 0.0139   & 0.0743   & 0.0175    \\\cline{2-8}
&GRU & 0.5496   & 0.8507   & 0.0505   & 0.0092   & 0.0831   & 0.0465    \\\cline{2-8}
\hline
\multirow{4}{*}{AI4I}
&XGB & 0.0537   & 0.0760   & 0.0280   & 0.0067   & 0.0857   & 0.0370    \\\cline{2-8}
&RF & 0.0322   & 0.0489   & 0.0264   & 0.0038   & 0.0706   & 0.1283    \\\cline{2-8}
&DT 

In [ ]:
for i, dataset in enumerate(data_characteristics):
    with open('evaluation/results_rob_all_' + dataset + '.pkl', 'rb') as f:
        results = pickle.load(f)
    if i == 0:
        print('Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\\\')
        print('\hline')
    print('\multirow{4}{*}{'+dataset+'}')
    df_all_rob_time =  pd.concat([df_all_rob_time,analyze_dataset_runtime_rob(results,dataset)])

Dataset & Algorithm & CE & ACE & LC & SC & LU & SU \\
\hline
\multirow{4}{*}{CIA}
&XGB & 0.0459   & 0.0608   & 0.0314   & 0.0110   & 0.0812   & 0.0635    \\\cline{2-8}
&RF & 0.0271   & 0.0384   & 0.0295   & 0.0067   & 0.0643   & 0.2057    \\\cline{2-8}
&DT & 0.0227   & 0.0277   & 0.0290   & 0.0062   & 0.0377   & 0.0082    \\\cline{2-8}
&GRU & 0.0315   & 0.0346   & 0.0316   & 0.0085   & 0.0528   & 0.0680    \\\cline{2-8}
\hline
\multirow{4}{*}{Azure}
&XGB & 0.5152   & 0.7737   & 0.0453   & 0.0103   & 0.1130   & 0.0595    \\\cline{2-8}
&RF & 0.4893   & 0.7454   & 0.0440   & 0.0070   & 0.0942   & 0.1963    \\\cline{2-8}
&DT & 0.4766   & 0.7329   & 0.0439   & 0.0067   & 0.0585   & 0.0085    \\\cline{2-8}
&GRU & 0.4728   & 0.7208   & 0.0458   & 0.0087   & 0.0781   & 0.0466    \\\cline{2-8}
\hline
\multirow{4}{*}{AI4I}
&XGB & 0.0539   & 0.0725   & 0.0281   & 0.0068   & 0.0854   & 0.0411    \\\cline{2-8}
&RF & 0.0319   & 0.0463   & 0.0264   & 0.0039   & 0.0712   & 0.1295    \\\cline{2-8}
&DT 

In [ ]:
df_all_stabilities.to_csv("evaluation/stability_all.csv", index=False)  
df_all_stability_time.to_csv("evaluation/stability_time.csv", index=False)  
df_all_rob_time.to_csv("evaluation/rob_time.csv", index=False)  
df_all_rob.to_csv("evaluation/rob_all.csv", index=False)  
